## Secrets Manager vs. Parameter Store

Applications need credentials at runtime — database passwords, API keys, OAuth tokens. Hard-code them in config and you have a leak waiting to happen. Two services hold them; the split is **rotation, cost, and size**.

### Secrets Manager

Stores credentials and returns them via a single `GetSecretValue` call. Two things make it more than a key-value store:

- **Rotation.** A **rotation Lambda** runs four steps — create a new credential in the target, update the secret, test it, retire the old one. Built-in rotation Lambdas exist for RDS (MySQL, PostgreSQL, Oracle, SQL Server, Aurora), Redshift, and DocumentDB; write a custom Lambda for anything else. Apps read the current value on each connection, so rotation needs **no restart or redeploy**.
- **Versioning.** Every value is versioned with staging labels (`AWSCURRENT`, `AWSPENDING`), so rotation can stage-and-promote safely.

### The comparison

| | Secrets Manager | SSM Parameter Store |
|---|---|---|
| Cost | $0.40/secret/mo + API | **free** (Standard); $0.05 (Advanced) |
| Rotation | **built-in** (Lambda) | none — manual only |
| Encryption | always KMS | optional KMS (`SecureString`) |
| Max size | 65 KB | 4 KB Std, 8 KB Advanced |
| RDS native integration | yes | no |

The rule: **rotation needed → Secrets Manager** (database creds, expiring API keys, OAuth secrets). **Plain, non-secret config → Parameter Store** (feature flags, ARNs, environment names) — free at Standard and integrated everywhere. Many architectures use **both**.